In [2]:
pip install streamlit

  Using cached streamlit-1.65.0-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pyarrow-25.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached starlette-1.7.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.32-py3-none-any

In [ ]:
# app.py

from datetime import date
import os
import tempfile
import streamlit as st
from src.speech_engine import speech_to_text, text_to_speech
from src.rag_engine import get_agent_executor
from src.vaccine_tracker import (
    build_vaccine_schedule,
    calculate_age,
    group_vaccines_by_status,
)

# Safe imports for modular components
try:
    from src.health_center import display_health_center_finder
except ImportError:
    display_health_center_finder = None

try:
    from src.pdf_generator import render_pdf_generator
except ImportError:
    render_pdf_generator = None

st.set_page_config(
    page_title="CNITS - Child Nutrition & Immunization Tracking System",
    page_icon="👶",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Initialize Session State for Authentication
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "user_name" not in st.session_state:
    st.session_state.user_name = "User"

# Custom CSS for vibrant cards, badges, and layout styling
st.markdown("""
<style>
.stApp {
    background: #F5FAF9;
    color: #153D3D;
}
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #064E4A, #003F40);
}
[data-testid="stSidebar"] * {
    color: #FFFFFF;
}
[data-testid="stSidebar"] .stRadio label {
    padding: 8px 10px;
}
h1, h2, h3 {
    color: #064E4A;
}
.block-container {
    padding-top: 1.5rem;
    padding-bottom: 2rem;
}
.stButton > button {
    background: #087F79;
    color: white;
    border: 1px solid #087F79;
    border-radius: 10px;
    min-height: 42px;
    font-weight: 600;
}
.stButton > button:hover {
    background: #05645F;
    color: white;
}
.stTextArea textarea, .stTextInput input {
    background: white;
    border-radius: 8px;
}
[data-testid="stMetric"] {
    background: white;
    padding: 16px;
    border-radius: 12px;
    border: 1px solid #D9EAE7;
    box-shadow: 0 2px 8px #064e4a0a;
}
[data-testid="stMetricValue"] {
    font-size: 1.35rem !important;
}
[data-testid="stMetricLabel"] {
    color: #315A59;
}
.hero {
    background: linear-gradient(110deg, #E8F8F3, #D4F0E5);
    padding: 28px;
    border-radius: 16px;
    border: 1px solid #D7EEE6;
    margin-bottom: 20px;
}
.panel {
    background: white;
    padding: 22px;
    border: 1px solid #DCEAE8;
    border-radius: 14px;
    margin-bottom: 16px;
}
.small-note {
    color: #496C69;
    font-size: 14px;
}
</style>
""", unsafe_allow_html=True)

# ----------------- AUTHENTICATION GATEWAY -----------------
def render_auth_page():
    st.markdown("<br><br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.markdown("""
        <div style="text-align: center; margin-bottom: 20px;">
            <h1>👶 Welcome to CNITS</h1>
            <p>Child Nutrition & Immunization Tracking System</p>
        </div>
        """, unsafe_allow_html=True)
        
        tab_signin, tab_signup = st.tabs(["🔐 Sign In", "📝 Sign Up"])
        
        with tab_signin:
            st.markdown("### Access Your Account")
            email = st.text_input("Email / Username", key="signin_email")
            password = st.text_input("Password", type="password", key="signin_pass")
            if st.button("Sign In to Workspace", use_container_width=True):
                if email and password:
                    st.session_state.authenticated = True
                    st.session_state.user_name = email.split("@")[0].capitalize()
                    st.rerun()
                else:
                    st.error("Please enter your credentials.")
                    
        with tab_signup:
            st.markdown("### Create New Account")
            new_name = st.text_input("Full Name", key="signup_name")
            new_email = st.text_input("Email Address", key="signup_email")
            new_pass = st.text_input("Create Password", type="password", key="signup_pass")
            if st.button("Register Account", use_container_width=True):
                if new_name and new_email and new_pass:
                    st.success("Account created successfully! Please switch to the Sign In tab.")
                else:
                    st.error("Please fill out all fields.")

if not st.session_state.authenticated:
    render_auth_page()
    st.stop()

# ----------------- TOP HEADER (Language & Profile) -----------------
header_col1, header_col2, header_col3 = st.columns([6, 2, 1])
with header_col2:
    selected_lang = st.selectbox(
        "Language", 
        ["🌐 English", "🌐 Yoruba", "🌐 Hausa", "🌐 Igbo"],
        label_visibility="collapsed"
    )
with header_col3:
    if st.button(f"👤 {st.session_state.user_name}", use_container_width=True):
        st.session_state.authenticated = False
        st.rerun()

# ----------------- SIDEBAR -----------------
with st.sidebar:
    st.markdown("# 👶 CNITS")
    st.caption("Child Nutrition & Immunization Tracking System")
    st.divider()

    page = st.radio(
        "Navigation",
        [
            "Home",
            "Nutrition AI",
            "Immunization Tracker",
            "Health Center Finder",
            "Growth & Health Passport",
            "Settings",
            "Help"
        ],
        label_visibility="collapsed",
        key="main_navigation"
    )

    st.divider()
    # Medical Disclaimer included directly in the sidebar
    st.info(
        "⚠️ **Medical Disclaimer:**\n"
        "CNITS is designed for tracking and educational purposes. "
        "Always consult a qualified healthcare provider for medical advice."
    )
    
    st.caption("💚 Healthy Children, Brighter Futures")

# HEADER BANNER
st.markdown("""
<div class="hero">
    <h1>🌱 Welcome to CNITS!</h1>
    <p>Together for healthier, stronger and brighter children.</p>
</div>
""", unsafe_allow_html=True)

# Initialize Agent
has_ai_credentials = all(
    os.getenv(key) for key in ("OPENAI_API_KEY", "EMBEDDING_API_KEY")
)
if has_ai_credentials and "agent" not in st.session_state:
    st.session_state.agent = get_agent_executor()

if "messages" not in st.session_state:
    st.session_state.messages = []

# HOME DASHBOARD
if page == "Home":
    st.subheader("Child Health Overview")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("💉 System Status", "Active" if has_ai_credentials else "Setup Key")
    c2.metric("📚 Vector DB", "Connected")
    c3.metric("🏥 Health Centers", "Loaded")
    c4.metric("🏆 Competition", "Finals 2026")

    st.markdown("### Your CNITS Workspace")
    st.caption("Choose a feature below to continue.")

    nav1, nav2, nav3, nav4 = st.columns(4)

    if nav1.button("🥗 Nutrition AI", use_container_width=True):
        st.session_state["main_navigation"] = "Nutrition AI"
        st.rerun()

    if nav2.button("💉 Immunization", use_container_width=True):
        st.session_state["main_navigation"] = "Immunization Tracker"
        st.rerun()

    if nav3.button("🏥 Health Centers", use_container_width=True):
        st.session_state["main_navigation"] = "Health Center Finder"
        st.rerun()

    if nav4.button("📈 Growth Passport", use_container_width=True):
        st.session_state["main_navigation"] = "Growth & Health Passport"
        st.rerun()

    st.markdown("""
    <div class="panel">
        <h3>💚 Healthy Today, Brighter Tomorrow</h3>
        <p>CNITS brings child nutrition, immunization and health record support into one digital workspace.</p>
        <p class="small-note">Designed to support caregivers, healthcare professionals and competition judges.</p>
    </div>
    """, unsafe_allow_html=True)

# NUTRITION AI (RAG & Voice Integration)
elif page == "Nutrition AI":
    st.markdown("## 🥗 Nutrition AI Assistant")
    st.write("Ask any nutrition question, describe a meal, or record your voice in English or local Nigerian languages.")

    if not has_ai_credentials:
        st.warning("Please configure your `OPENAI_API_KEY` and `EMBEDDING_API_KEY` in your `.env` file.")

    left, right = st.columns([3, 2], gap="large")

    with left:
        recording = st.audio_input("🎙️ Record your question")

        if recording is not None:
            if st.button("Convert Speech to Text"):
                temp_path = None
                try:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
                        f.write(recording.getvalue())
                        temp_path = f.name

                    with st.spinner("Recognizing speech..."):
                        transcript = speech_to_text(temp_path)

                    st.session_state["nutrition_question"] = transcript
                    st.success("Transcription complete.")
                except Exception as error:
                    st.error(f"Speech recognition failed: {error}")
                finally:
                    if temp_path and os.path.exists(temp_path):
                        os.remove(temp_path)

        user_input = st.text_area(
            "Your question or meal description",
            placeholder="Example: What local foods can I give my 6-month-old baby in Nigeria?",
            key="nutrition_question",
            height=130
        )

        col_btn1, col_btn2 = st.columns(2)
        with col_btn1:
            evaluate_btn = st.button("✨ Evaluate with AI", use_container_width=True, disabled=not has_ai_credentials)
        with col_btn2:
            tts_btn = st.button("🔊 Read Aloud", use_container_width=True)

        if evaluate_btn and user_input.strip():
            with st.spinner("Analyzing NPHCDA & Health Guidelines..."):
                config = {"configurable": {"thread_id": "streamlit_user_session"}}
                prompt_instruction = (
                    f"Detect the language of the following user question and reply in that exact same language. "
                    f"User question: {user_input}"
                )
                response = st.session_state.agent.invoke(
                    {"messages": [("user", prompt_instruction)]},
                    config=config
                )
                bot_reply = response["messages"][-1].content
                st.markdown("### 🤖 AI Response:")
                st.success(bot_reply)
                st.session_state["last_bot_reply"] = bot_reply

        if tts_btn:
            text_to_read = st.session_state.get("last_bot_reply", user_input)
            if text_to_read.strip():
                audio_path = None
                try:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
                        audio_path = f.name
                    text_to_speech(text_to_read, audio_path)
                    with open(audio_path, "rb") as f:
                        st.audio(f.read(), format="audio/mp3")
                except Exception as error:
                    st.error(f"Text-to-speech failed: {error}")
                finally:
                    if audio_path and os.path.exists(audio_path):
                        os.remove(audio_path)
            else:
                st.warning("No text available to read.")

    with right:
        st.markdown("""
        <div class="panel">
            <h3>👶 Get Nutrition Support</h3>
            <p>CNITS nutrition guidance helps users:</p>
            <p>✓ Review meal information against NFCMS data</p>
            <p>✓ Identify possible nutrient gaps</p>
            <p>✓ Explore local Nigerian food pairings (crayfish, soybeans, pap)</p>
            <p>✓ Interact through voice or text in local languages</p>
        </div>
        """, unsafe_allow_html=True)

# IMMUNIZATION TRACKER
elif page == "Immunization Tracker":
    st.header("💉 Smart Immunization Schedule Tracker")
    st.write("Calculate your child's vaccination timeline based on the official Nigerian NPHCDA schedule.")

    col1, col2 = st.columns(2)
    with col1:
        dob = st.date_input("Child's Date of Birth", value=date(2026, 1, 1))
    with col2:
        as_of = st.date_input("Tracking Reference Date", value=date.today())

    if dob > as_of:
        st.error("Date of birth cannot be in the future.")
    else:
        age_info = calculate_age(dob, as_of)
        st.info(f"**Current Age:** {age_info['years']} yrs, {age_info['total_months'] % 12} mos ({age_info['total_weeks']} weeks / {age_info['total_days']} days)")

        records = build_vaccine_schedule(dob, as_of=as_of)
        grouped = group_vaccines_by_status(records)

        m1, m2, m3, m4 = st.columns(4)
        m1.metric("🔴 Overdue", len(grouped["overdue"]))
        m2.metric("🟢 Due Now", len(grouped["due_now"]))
        m3.metric("🔵 Upcoming", len(grouped["upcoming"]))
        m4.metric("✅ Completed", len(grouped["completed"]))

        st.divider()
        display_data = []
        for r in records:
            display_data.append({
                "Vaccine": r["vaccine"],
                "Milestone": r["target_age_label"],
                "Due Date": r["due_date_label"],
                "Status": r["status"],
                "Dosage": r["dosage"],
                "Route": r["route"],
                "Site": r["site"]
            })
        st.dataframe(display_data, use_container_width=True, hide_index=True)

# HEALTH CENTERS
elif page == "Health Center Finder":
    if display_health_center_finder:
        display_health_center_finder()
    else:
        st.info("Health center finder module loading...")

# GROWTH & HEALTH PASSPORT
elif page == "Growth & Health Passport":
    if render_pdf_generator:
        render_pdf_generator()
    else:
        st.info("PDF generator module loading...")

# SETTINGS
elif page == "Settings":
    st.header("⚙️ Settings")
    st.write("Language preferences, API configuration, and user settings.")

# HELP
elif page == "Help":
    st.header("❓ Help & Support Center")
    st.markdown("""
        <div style="background-color: #ffffff; padding: 24px; border-radius: 16px; border: 1px solid #e2e8f0;">
            <h3 style="color: #064E4A; margin-top: 0;">📌 Quick User Guide</h3>
            <ul>
                <li><b>Nutrition AI:</b> Ask questions or record voice notes to get localized nutritional advice.</li>
                <li><b>Immunization Tracker:</b> Automatically calculates NPHCDA routine childhood immunization schedules.</li>
                <li><b>Health Center Finder:</b> Locate nearby primary healthcare facilities instantly.</li>
                <li><b>Growth & Health Passport:</b> Export downloadable digital health passport PDFs.</li>
            </ul>
        </div>
    """, unsafe_allow_html=True)

# FOOTER DISCLAIMER
st.markdown("---")
st.caption(
    "⚠️ **Disclaimer:** CNITS helps you track your child's nutrition, immunization, and health records — all in one place. "
    "Always consult a qualified healthcare professional for formal pediatric or medical guidance."
)# app.py

from datetime import date
import os
import tempfile
import streamlit as st
from src.speech_engine import speech_to_text, text_to_speech
from src.rag_engine import get_agent_executor
from src.vaccine_tracker import (
    build_vaccine_schedule,
    calculate_age,
    group_vaccines_by_status,
)

# Safe imports for modular components
try:
    from src.health_center import display_health_center_finder
except ImportError:
    display_health_center_finder = None

try:
    from src.pdf_generator import render_pdf_generator
except ImportError:
    render_pdf_generator = None

st.set_page_config(
    page_title="CNITS - Child Nutrition & Immunization Tracking System",
    page_icon="👶",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Initialize Session State for Authentication
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "user_name" not in st.session_state:
    st.session_state.user_name = "User"

# Custom CSS for vibrant cards, badges, and layout styling
st.markdown("""
<style>
.stApp {
    background: #F5FAF9;
    color: #153D3D;
}
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #064E4A, #003F40);
}
[data-testid="stSidebar"] * {
    color: #FFFFFF;
}
[data-testid="stSidebar"] .stRadio label {
    padding: 8px 10px;
}
h1, h2, h3 {
    color: #064E4A;
}
.block-container {
    padding-top: 1.5rem;
    padding-bottom: 2rem;
}
.stButton > button {
    background: #087F79;
    color: white;
    border: 1px solid #087F79;
    border-radius: 10px;
    min-height: 42px;
    font-weight: 600;
}
.stButton > button:hover {
    background: #05645F;
    color: white;
}
.stTextArea textarea, .stTextInput input {
    background: white;
    border-radius: 8px;
}
[data-testid="stMetric"] {
    background: white;
    padding: 16px;
    border-radius: 12px;
    border: 1px solid #D9EAE7;
    box-shadow: 0 2px 8px #064e4a0a;
}
[data-testid="stMetricValue"] {
    font-size: 1.35rem !important;
}
[data-testid="stMetricLabel"] {
    color: #315A59;
}
.hero {
    background: linear-gradient(110deg, #E8F8F3, #D4F0E5);
    padding: 28px;
    border-radius: 16px;
    border: 1px solid #D7EEE6;
    margin-bottom: 20px;
}
.panel {
    background: white;
    padding: 22px;
    border: 1px solid #DCEAE8;
    border-radius: 14px;
    margin-bottom: 16px;
}
.small-note {
    color: #496C69;
    font-size: 14px;
}
</style>
""", unsafe_allow_html=True)

# ----------------- AUTHENTICATION GATEWAY -----------------
def render_auth_page():
    st.markdown("<br><br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.markdown("""
        <div style="text-align: center; margin-bottom: 20px;">
            <h1>👶 Welcome to CNITS</h1>
            <p>Child Nutrition & Immunization Tracking System</p>
        </div>
        """, unsafe_allow_html=True)
        
        tab_signin, tab_signup = st.tabs(["🔐 Sign In", "📝 Sign Up"])
        
        with tab_signin:
            st.markdown("### Access Your Account")
            email = st.text_input("Email / Username", key="signin_email")
            password = st.text_input("Password", type="password", key="signin_pass")
            if st.button("Sign In to Workspace", use_container_width=True):
                if email and password:
                    st.session_state.authenticated = True
                    st.session_state.user_name = email.split("@")[0].capitalize()
                    st.rerun()
                else:
                    st.error("Please enter your credentials.")
                    
        with tab_signup:
            st.markdown("### Create New Account")
            new_name = st.text_input("Full Name", key="signup_name")
            new_email = st.text_input("Email Address", key="signup_email")
            new_pass = st.text_input("Create Password", type="password", key="signup_pass")
            if st.button("Register Account", use_container_width=True):
                if new_name and new_email and new_pass:
                    st.success("Account created successfully! Please switch to the Sign In tab.")
                else:
                    st.error("Please fill out all fields.")

if not st.session_state.authenticated:
    render_auth_page()
    st.stop()

# ----------------- TOP HEADER (Language & Profile) -----------------
header_col1, header_col2, header_col3 = st.columns([6, 2, 1])
with header_col2:
    selected_lang = st.selectbox(
        "Language", 
        ["🌐 English", "🌐 Yoruba", "🌐 Hausa", "🌐 Igbo"],
        label_visibility="collapsed"
    )
with header_col3:
    if st.button(f"👤 {st.session_state.user_name}", use_container_width=True):
        st.session_state.authenticated = False
        st.rerun()

# ----------------- SIDEBAR -----------------
with st.sidebar:
    st.markdown("# 👶 CNITS")
    st.caption("Child Nutrition & Immunization Tracking System")
    st.divider()

    page = st.radio(
        "Navigation",
        [
            "Home",
            "Nutrition AI",
            "Immunization Tracker",
            "Health Center Finder",
            "Growth & Health Passport",
            "Settings",
            "Help"
        ],
        label_visibility="collapsed",
        key="main_navigation"
    )

    st.divider()
    # Medical Disclaimer included directly in the sidebar
    st.info(
        "⚠️ **Medical Disclaimer:**\n"
        "CNITS is designed for tracking and educational purposes. "
        "Always consult a qualified healthcare provider for medical advice."
    )
    
    st.caption("💚 Healthy Children, Brighter Futures")

# HEADER BANNER
st.markdown("""
<div class="hero">
    <h1>🌱 Welcome to CNITS!</h1>
    <p>Together for healthier, stronger and brighter children.</p>
</div>
""", unsafe_allow_html=True)

# Initialize Agent
has_ai_credentials = all(
    os.getenv(key) for key in ("OPENAI_API_KEY", "EMBEDDING_API_KEY")
)
if has_ai_credentials and "agent" not in st.session_state:
    st.session_state.agent = get_agent_executor()

if "messages" not in st.session_state:
    st.session_state.messages = []

# HOME DASHBOARD
if page == "Home":
    st.subheader("Child Health Overview")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("💉 System Status", "Active" if has_ai_credentials else "Setup Key")
    c2.metric("📚 Vector DB", "Connected")
    c3.metric("🏥 Health Centers", "Loaded")
    c4.metric("🏆 Competition", "Finals 2026")

    st.markdown("### Your CNITS Workspace")
    st.caption("Choose a feature below to continue.")

    nav1, nav2, nav3, nav4 = st.columns(4)

    if nav1.button("🥗 Nutrition AI", use_container_width=True):
        st.session_state["main_navigation"] = "Nutrition AI"
        st.rerun()

    if nav2.button("💉 Immunization", use_container_width=True):
        st.session_state["main_navigation"] = "Immunization Tracker"
        st.rerun()

    if nav3.button("🏥 Health Centers", use_container_width=True):
        st.session_state["main_navigation"] = "Health Center Finder"
        st.rerun()

    if nav4.button("📈 Growth Passport", use_container_width=True):
        st.session_state["main_navigation"] = "Growth & Health Passport"
        st.rerun()

    st.markdown("""
    <div class="panel">
        <h3>💚 Healthy Today, Brighter Tomorrow</h3>
        <p>CNITS brings child nutrition, immunization and health record support into one digital workspace.</p>
        <p class="small-note">Designed to support caregivers, healthcare professionals and competition judges.</p>
    </div>
    """, unsafe_allow_html=True)

# NUTRITION AI (RAG & Voice Integration)
elif page == "Nutrition AI":
    st.markdown("## 🥗 Nutrition AI Assistant")
    st.write("Ask any nutrition question, describe a meal, or record your voice in English or local Nigerian languages.")

    if not has_ai_credentials:
        st.warning("Please configure your `OPENAI_API_KEY` and `EMBEDDING_API_KEY` in your `.env` file.")

    left, right = st.columns([3, 2], gap="large")

    with left:
        recording = st.audio_input("🎙️ Record your question")

        if recording is not None:
            if st.button("Convert Speech to Text"):
                temp_path = None
                try:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
                        f.write(recording.getvalue())
                        temp_path = f.name

                    with st.spinner("Recognizing speech..."):
                        transcript = speech_to_text(temp_path)

                    st.session_state["nutrition_question"] = transcript
                    st.success("Transcription complete.")
                except Exception as error:
                    st.error(f"Speech recognition failed: {error}")
                finally:
                    if temp_path and os.path.exists(temp_path):
                        os.remove(temp_path)

        user_input = st.text_area(
            "Your question or meal description",
            placeholder="Example: What local foods can I give my 6-month-old baby in Nigeria?",
            key="nutrition_question",
            height=130
        )

        col_btn1, col_btn2 = st.columns(2)
        with col_btn1:
            evaluate_btn = st.button("✨ Evaluate with AI", use_container_width=True, disabled=not has_ai_credentials)
        with col_btn2:
            tts_btn = st.button("🔊 Read Aloud", use_container_width=True)

        if evaluate_btn and user_input.strip():
            with st.spinner("Analyzing NPHCDA & Health Guidelines..."):
                config = {"configurable": {"thread_id": "streamlit_user_session"}}
                prompt_instruction = (
                    f"Detect the language of the following user question and reply in that exact same language. "
                    f"User question: {user_input}"
                )
                response = st.session_state.agent.invoke(
                    {"messages": [("user", prompt_instruction)]},
                    config=config
                )
                bot_reply = response["messages"][-1].content
                st.markdown("### 🤖 AI Response:")
                st.success(bot_reply)
                st.session_state["last_bot_reply"] = bot_reply

        if tts_btn:
            text_to_read = st.session_state.get("last_bot_reply", user_input)
            if text_to_read.strip():
                audio_path = None
                try:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
                        audio_path = f.name
                    text_to_speech(text_to_read, audio_path)
                    with open(audio_path, "rb") as f:
                        st.audio(f.read(), format="audio/mp3")
                except Exception as error:
                    st.error(f"Text-to-speech failed: {error}")
                finally:
                    if audio_path and os.path.exists(audio_path):
                        os.remove(audio_path)
            else:
                st.warning("No text available to read.")

    with right:
        st.markdown("""
        <div class="panel">
            <h3>👶 Get Nutrition Support</h3>
            <p>CNITS nutrition guidance helps users:</p>
            <p>✓ Review meal information against NFCMS data</p>
            <p>✓ Identify possible nutrient gaps</p>
            <p>✓ Explore local Nigerian food pairings (crayfish, soybeans, pap)</p>
            <p>✓ Interact through voice or text in local languages</p>
        </div>
        """, unsafe_allow_html=True)

# IMMUNIZATION TRACKER
elif page == "Immunization Tracker":
    st.header("💉 Smart Immunization Schedule Tracker")
    st.write("Calculate your child's vaccination timeline based on the official Nigerian NPHCDA schedule.")

    col1, col2 = st.columns(2)
    with col1:
        dob = st.date_input("Child's Date of Birth", value=date(2026, 1, 1))
    with col2:
        as_of = st.date_input("Tracking Reference Date", value=date.today())

    if dob > as_of:
        st.error("Date of birth cannot be in the future.")
    else:
        age_info = calculate_age(dob, as_of)
        st.info(f"**Current Age:** {age_info['years']} yrs, {age_info['total_months'] % 12} mos ({age_info['total_weeks']} weeks / {age_info['total_days']} days)")

        records = build_vaccine_schedule(dob, as_of=as_of)
        grouped = group_vaccines_by_status(records)

        m1, m2, m3, m4 = st.columns(4)
        m1.metric("🔴 Overdue", len(grouped["overdue"]))
        m2.metric("🟢 Due Now", len(grouped["due_now"]))
        m3.metric("🔵 Upcoming", len(grouped["upcoming"]))
        m4.metric("✅ Completed", len(grouped["completed"]))

        st.divider()
        display_data = []
        for r in records:
            display_data.append({
                "Vaccine": r["vaccine"],
                "Milestone": r["target_age_label"],
                "Due Date": r["due_date_label"],
                "Status": r["status"],
                "Dosage": r["dosage"],
                "Route": r["route"],
                "Site": r["site"]
            })
        st.dataframe(display_data, use_container_width=True, hide_index=True)

# HEALTH CENTERS
elif page == "Health Center Finder":
    if display_health_center_finder:
        display_health_center_finder()
    else:
        st.info("Health center finder module loading...")

# GROWTH & HEALTH PASSPORT
elif page == "Growth & Health Passport":
    if render_pdf_generator:
        render_pdf_generator()
    else:
        st.info("PDF generator module loading...")

# SETTINGS
elif page == "Settings":
    st.header("⚙️ Settings")
    st.write("Language preferences, API configuration, and user settings.")

# HELP
elif page == "Help":
    st.header("❓ Help & Support Center")
    st.markdown("""
        <div style="background-color: #ffffff; padding: 24px; border-radius: 16px; border: 1px solid #e2e8f0;">
            <h3 style="color: #064E4A; margin-top: 0;">📌 Quick User Guide</h3>
            <ul>
                <li><b>Nutrition AI:</b> Ask questions or record voice notes to get localized nutritional advice.</li>
                <li><b>Immunization Tracker:</b> Automatically calculates NPHCDA routine childhood immunization schedules.</li>
                <li><b>Health Center Finder:</b> Locate nearby primary healthcare facilities instantly.</li>
                <li><b>Growth & Health Passport:</b> Export downloadable digital health passport PDFs.</li>
            </ul>
        </div>
    """, unsafe_allow_html=True)

# FOOTER DISCLAIMER
st.markdown("---")
st.caption(
    "⚠️ **Disclaimer:** CNITS helps you track your child's nutrition, immunization, and health records — all in one place. "
    "Always consult a qualified healthcare professional for formal pediatric or medical guidance."
)

In [1]:
import streamlit as st
st.set_page_config(
        page_title="CNTIS - Child Nutrition & Immunization Tracking System",
        page_icon="👶",
        layout ="wide"
    )

# Custom CSS for vibrant cards, badges, and layout styling
st.markdown("""
<style>
/* Main page background */
.stApp {
    background-color: #F3F8F7;
    color: #173B3A;
}

/* Sidebar */
[data-testid="stSidebar"] {
    background-color: #064E4A;
}

/* Sidebar text */
[data-testid="stSidebar"] * {
    color: #FFFFFF;
}

/* Main headings */
h1, h2, h3 {
    color: #115E59;
}

/* Buttons */
.stButton > button {
    background-color: #0F766E;
    color: white;
    border: none;
    border-radius: 10px;
    padding: 0.6rem 1rem;
    font-weight: 600;
}

/* Button hover */
.stButton > button:hover {
    background-color: #115E59;
    color: white;
    border: 1px solid #5EEAD4;
}

/* Input fields and text areas */
.stTextInput input,
.stNumberInput input,
.stTextArea textarea {
    background-color: white;
    border-radius: 8px;
}

/* Tabs, if used elsewhere */
[data-baseweb="tab-list"] {
    gap: 8px;
}
</style>
""", unsafe_allow_html=True)

# Sidebar branding
with st.sidebar:
    st.markdown("# 👶 CNITS")
    st.caption("Child Nutrition & Immunization Tracking System")
    st.divider()

    st.markdown("### NAVIGATION")

    page = st.radio(
        [   "Home",
            "Nutrition AI",
            "Immunization Tracker",
            "Health Center Finder",
            "Growth & Health Passport"
        ],
        label_visibility="visible"
    )

    st.divider()
    st.caption("Healthy Children, Brighter Futures 💚")
    st.caption("Nutrition • Immunization • Child Health")

# App Header
st.markdown("<h1 class='main-title'>👶 Child Nutrition & Immunization Tracking System (CNITS)</h1>", unsafe_allow_html=True)
st.markdown("<p class='sub-text'>A comprehensive digital health application to track child nutrition, immunization schedules, and health records.</p>", unsafe_allow_html=True)
st.divider()

# st.title("👶Child Nutrition & Immunization Tracking System (CNITS)")
# st.markdown("A comprehensive digital health application to track child nutrition, immunization schedules, and health records.")

tab1, tab2, tab3, tab4 = st.tabs([
    "🥗Nutrition AI",
    "💉Immunization Tracker & Reminders",
    "🏥Health Center Finder",
    "📈 Growth & Health Passport"
]) 
with tab1:
    st.header("Nutrition AI")
    st.markdown("Analyze meals instantly using AI")
    child_age = st.number_input("Child's age (months)", min_value=6, max_value=59)
    meal = st.text_area("Enter the child's meal")

    if st.button("Evaluate meal"):
        st.info("Nutrition evaluation will appear here.")
with tab2:
    st.header("Immunization Tracker & Reminders")
    birth_date = st.date_input("Enter Child's Birthdate:")
    col1, col2,col3 = st.columns(3)
    
    with col1:
        st.metric("Overdue", "0")
    with col2:
        st.metric("Due Now", "0")
    with col3:
        st.metric("Upcoming", "0")
    
    st.subheader("Vaccine Schedule")
    st.info ("Vaccine status will be calculated from the child's birthdate using NPHCDA guidelines")
    st.button("Download Calendar Reminders")
    # vaccine_tracker.py incoming here
with tab3:
    st.header("Health Center Finder")
    st.markdown("Locate nearby primary healthcare centers.")
    # health_center.py incoming here
    location = st.text_input(
        "Enter your location",
        placeholder="e.g. Ikorodu, Lagos"
        )
    if st.button("Find Health Centers"):
            st.subheader("Nearby Primary Healthcare Centers")
            st.info("Health center will be connected to the health_center.py module.")
            st.write("📍 Location:", location)
            st.write("🏥 Nearest healthcare center will appear here.")
with tab4:
    st.header("Growth & Health Passport")
    st.markdown("Track height/ weight against WHO standards and download your PDF Health Passport.")
    # pdf_generator.py incoming here    
    with col1:
        height = st.number_input(
            "Height (cm)", min_value=40.0, max_value =150.0)

    with col2: 
        weight = st.number_input("Weight(kg)", min_value=1.0, max_value=50.0)
        if st.button("Record Growth"):
            st.success("Growth record saved.")

    st.subheader("Growth History")

    st.info("WHO growth comparison chart will appear here.")

    if st.button("📄 Download Health Passport"):
        st.info("PDF Health Passport will be generated by pdf_generator.py.")


ModuleNotFoundError: No module named 'streamlit'